# Extraction Pipeline v10

**Pipeline steps:**
1. Variable Detection — role labelling (IV / DV / Moderator / Mediator)
2. Hierarchy & Normalization — split combined concepts, detect hierarchy / levels
3. Relevant Sentence Extraction
4. Relationship Extraction — directional / correlational / moderation
5. Validation Validate directional / correlational / moderation 

Step1, 5 -> GPT-5.4 
Step2,3,4 -> GPT-5.2

In [ ]:
# Cell 1 — Imports

import json
import re
import os
import warnings
from typing import Dict, List, Tuple
from datetime import datetime
from textwrap import dedent
import pandas as pd
from openai import OpenAI
from json_repair import repair_json

warnings.filterwarnings("ignore")
print("Imports successful.")

In [ ]:
# Cell 2 — OpenAI client & model constants

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise ValueError("Set OPENAI_API_KEY in environment first.")

client = OpenAI(api_key=OPENAI_API_KEY)

MODEL_SIMPLE  = "gpt-4.1"
MODEL_COMPLEX = "gpt-5.2"

print(f"OpenAI client initialised")
print(f"MODEL_SIMPLE  : {MODEL_SIMPLE}")
print(f"MODEL_COMPLEX : {MODEL_COMPLEX}")

In [ ]:
# Cell 3 — Abstract cleaning utility

_CLEAN_RE = re.compile(
    r"\s*\(PsycInfo Database Record.*$"
    r"|\s*\(PsycINFO Database Record.*$",
    re.DOTALL | re.IGNORECASE,
)

# Impact statement section patterns — everything from the marker to end of string
_IMPACT_RE = re.compile(
    r"\s*(Impact Statement|Public Significance Statement|Public Significance|"
    r"Practitioner Points|Educational Impact and Implications Statement|"
    r"Translational Significance|Clinical Impact|Lay Summary|Public Interest Summary|"
    r"What is the public significance|What does this article add)"
    r"[:\-\s].*$",
    re.DOTALL | re.IGNORECASE,
)


def clean_abstract(raw: str) -> str:
    """Remove trailing PsycInfo boilerplate and any Impact Statement section."""
    text = _CLEAN_RE.sub("", raw).strip()
    text = _IMPACT_RE.sub("", text).strip()
    return text


print("clean_abstract() defined.")

In [ ]:
# Cell 4 — Shared helper functions (Responses API)

MAX_RETRIES = 3
RETRY_DELAY_SEC = 2


def call_openai_with_json_simple(prompt_blocks, model=MODEL_SIMPLE):
    """Call OpenAI Responses API with gpt-4.1 (no reasoning)."""
    response = client.responses.create(
        model=model,
        input=prompt_blocks,
        temperature=0,
        text={"format": {"type": "text"}},
        tools=[],
        store=True,
    )
    return {
        "content": response.output_text,
        "raw_response": response,
        "usage": getattr(response, "usage", None),
    }


def call_openai_with_json_complex(prompt_blocks, model=MODEL_COMPLEX):
    """Call OpenAI Responses API with chatgpt-5.2 (with reasoning)."""
    response = client.responses.create(
        model=model,
        input=prompt_blocks,
        text={"format": {"type": "text"}},
        reasoning={"effort": "low", "summary": "auto"},
        tools=[],
        store=True,
    )
    return {
        "content": response.output_text,
        "raw_response": response,
        "usage": getattr(response, "usage", None),
    }


def extract_json_str(text: str) -> str:
    """Extract JSON string from markdown code blocks or raw text."""
    if not isinstance(text, str) or not text.strip():
        return ""
    json_block_match = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', text, re.DOTALL)
    if json_block_match:
        return json_block_match.group(1).strip()
    json_match = re.search(r'\{.*\}', text, re.DOTALL)
    if json_match:
        return json_match.group(0).strip()
    return text.strip()


def parse_json_response(response_dict: dict) -> dict:
    """Parse JSON from wrapped Responses API response dict."""
    text = response_dict.get("content", "")
    json_str = extract_json_str(text)
    try:
        return json.loads(json_str)
    except json.JSONDecodeError:
        pass
    try:
        repaired = repair_json(json_str)
        return json.loads(repaired)
    except Exception as e:
        raise ValueError(f"Failed to parse JSON after repair: {e}\nOriginal: {json_str[:500]}")


def extract_token_usage(response_dict: dict) -> Dict[str, int]:
    """Extract token usage from Responses API response dict."""
    info = {"input_tokens": 0, "output_tokens": 0, "total_tokens": 0}
    try:
        usage = response_dict.get("usage")
        if usage is not None:
            info["input_tokens"] = getattr(usage, "input_tokens", 0)
            info["output_tokens"] = getattr(usage, "output_tokens", 0)
            info["total_tokens"] = info["input_tokens"] + info["output_tokens"]
    except Exception:
        pass
    return info


print("Helper functions defined (Responses API + JSON repair).")

In [ ]:
# Cell 5 — Step 1: Variable Detection

def step1_detect_variables(
    abstract: str
) -> Tuple[dict, Dict[str, int]]:
    """
    Detect variables and label their roles (IV, DV, Moderator, Mediator).
    Uses developer message + structured user message.
    Uses gpt-5.4 via Responses API.
    """

    developer_content = dedent("""\
        You are a scientific-text analyst specializing in variable detection in research abstracts.
        Extract only true study variables explicitly investigated, measured, categorized, manipulated, or reported as substantive constructs in the abstract. Minimize false positives. Do not include generic outcomes, contextual phrases, narrative descriptions, methodological wording, summary language, or overly broad fragments unless they are clearly treated as variables in the study.
        When identifying variables, prefer specific constructs or named categories over umbrella phrases when both appear. Avoid including text spans that are merely explanatory, incidental, or derivative of another listed variable.
        Success criteria: include a text span only if the abstract presents it as a study variable through direct investigation, measurement, categorization, manipulation, or substantive reporting. If a term is ambiguous, exclude it unless the abstract clearly treats it as a variable. When both a broad label and its explicit named subcategories appear, include the subcategories and include the broad label only if the abstract itself treats that label as a reported variable or category set.
    """)

    user_content = dedent(f"""\
        <task>
        Your task is to extract the variable relationships from the abstract. Identify and label which variables are the independent variables (IVs), dependent variables (DVs), moderators, and mediators (if any).
        </task>

        <context>
        ABSTRACT:
        {abstract}
        </context>

        <output_format>
        Return a JSON object with this exact structure:

        {{
            "independent_variables": ["IV 1", "IV 2"],
            "dependent_variables": ["DV 1", "DV 2"],
            "moderators": ["Mod 1"],
            "mediators": ["Med 1"],
            "final_variable_list": ["var 1", "var 2", "var 3"]
        }}
        </output_format>

        <rules>
        Rules for final_variable_list:
        - Must be the union of all variables across all four role lists
        - Each variable appears only once
        - If no moderators or mediators exist, return empty lists for those fields

        Return ONLY the JSON object, no additional text.
        </rules>
    """)

    prompt_blocks = [
        {
            "role": "developer",
            "content": developer_content
        },
        {
            "role": "user",
            "content": user_content
        }
    ]

    response_dict = call_openai_with_json_complex(prompt_blocks=prompt_blocks)
    tokens = extract_token_usage(response_dict)
    result = parse_json_response(response_dict)

    return result, tokens

print("Step 1: step1_detect_variables() defined")

In [ ]:
# Cell 6 — Step 2: Hierarchy & Normalization

def step2_hierarchy_and_normalize(
    abstract: str,
    step1_result: dict
) -> Tuple[dict, Dict[str, int]]:
    """
    Split combined concepts, detect hierarchy/levels, normalize variable list.
    Takes Step 1 output (with IV/DV/Mod/Med roles) and the abstract.
    Uses gpt-5.2 via Responses API.
    """

    step1_json = json.dumps(step1_result, ensure_ascii=False, indent=2)

    developer_content = dedent("""\
        You are an expert scientific-text analyst specializing in identifying hierarchical relationships and normalizing variables in research abstracts.
        <definitions>
        WHAT IS A VALID HIERARCHY?
        CASE 1: Construct decomposition
        The abstract identifies, distinguishes, or analyzes components or subthemes of a broader concept, AND identifying those components is the main reported finding (typical in qualitative / mixed-methods studies).
        NOT Case 1:
        • Components mentioned only for definition or background
        • Appositional phrases ("X, including A, B, C") that merely list what something consists of without treating each as a reported finding
        Example:
        - Big Five personality -> Extraversion
        - Theme -> Subtheme
        - “we constructed X themes”
        - “the analysis revealed X themes/subthemes”
        - “we identified X themes”
        - “thematic analysis showed…”
        CASE 2: Abstract-to-specific relationship
        The abstract first mentions a broad construct-level variable/construct, AND THEN specifies lower-level dimensions/indicators (LV1, LV2... affect that same outcome).
        IN CASE 2, BOTH conditions must be met:
       (a) The study measures/analyzes several characteristics/aspects of a given construct, and these aspects are then each examined in relation to other variables (could be validated/null effect/or just propose or hypothesised). However, do not invent an overarching higher-level variable yourself; the higher-level variable must be explicitly mentioned in the original text. Therefore, you may not extract a shared head noun (the core noun phrase) from multiple variable names and treat it as a higher-level construct (e.g., extracting “depictive gestures” from “dynamic/nondynamic depictive gestures” and coding it as the higher-level variable).
        (b) In the abstract, each LV appears in at least one directional/correlational/moderation relationship with other variables.
        RELAXATION RULES:
        HV does not necessarily need to be a measured variables or in the model. It could also be the construct umbrella.
        You should stick to the abstract and check whether it mentioned how HV relate to other variable or how the study test the relationship of HV.
        The abstract might not claim an explicit decomposition of HV to LV1, LV2, LV3. You are allowed to infer implicit decomposition between HV and LV1, LV2, through your knowledge.
        Example 1:
        If the abstract first states a general relationship such as "this study examines how personality influences leadership," and then discusses several personality dimensions such as extraversion, agreeableness, and conscientiousness, then code:
        Personality -> Extraversion
        Personality -> Agreeableness
        Personality -> Conscientiousness
        Example 2:
        If the abstract states that the study examines how training affects spelling memory, and then measures spelling memory through several indicators, then code:
        Spelling memory -> Short-term spelling performance
        Spelling memory -> Long-term spelling performance
        Spelling memory -> Stability of spelling knowledge
        </definitions>
        <rules>
        Different levels of the same variable: Do not code a hierarchical relationship when a variable is only divided into different levels of the same variable.
        A variable may be divided only into different levels of the same variable, not into different subcomponents. The rule for judging whether something is "different levels of the same variable" is: does this component actually vary or get manipulated/measured in this study? For example, self-esteem (high/low) are levels, because "high self-esteem" is not itself a component that is manipulated or measured for variation.
        This SHOULD NOT BE CODED AS A HIERARCHY. A hierarchy requires meaningful subcomponents that can vary independently. For example, extraversion can vary independently, and spelling memory can be measured by different indicators.
        Code format for levels:
        Variable (level 1, level 2)
        Workflow:
        1. Check for combined concepts
        If a phrase expresses relationships among multiple concepts (for example: interaction, association, effect, "A and B", impact of A on B), split it into the individual concepts.
        Be cautious! You should only split combined concepts.
        2. Identify the hierarchical relationships and levels in the abstract.
        If you spot a hierarchical relationship, check whether both the higher-level variable and all lower-level variables appear in the variable list. If not, you should add variables from the abstract if they are included in hierarchical relationship.
        3. Coding the hierarchy and levels. The coding rules are as follows:
        For hierarchy relationships, put the upper-level variable in "variable" and lower-level variables in "components", and include the lower-level variables in "final_variable_list" as well.
        For levels, do NOT put different levels in components, do NOT put different levels in final_variable_list. Only code as variable (level1, level2).
        Do NOT change a variable's role position. If a variable is an Independent variable in the input variable list, do not move it to moderation or other positions.
        4. Final Check
        - Make sure all hierarchical relationships in the abstract are coded, and both the higher-level variable and all lower-level variables in each hierarchy relationship are added into the variable list.
        - Check again whether each hierarchy relationship meets either CASE 1 or CASE 2. If not, delete that hierarchy relationship.
        - Check whether variables coded as hierarchy sub-levels actually have manipulated or observed variation in the text. If there is no actual variation, reconsider whether it should be treated as a different level.
        Use exact variable names where possible.
        Return only valid JSON.
        </rules>
    """)

    user_content = dedent(f"""\
        <task>
        You are given an abstract and a variable list.
        Normalize the variable list, detect valid hierarchy relationships, and distinguish hierarchy from different levels of the same variable.
        </task>

        <output_format>
        Return a JSON object with this exact structure:

        {{
          "hierarchy": {{
            "hierarchical_relationships": [
              {{
                "higher_level_variable": "",
                "lower_level_variable": ""
              }}
            ]
          }},
          "final_variable_list": ["", ""],
          "independent_variables": [
            {{
              "variable": "",
              "components": ["", ""]
            }}
          ],
          "dependent_variables": [
            {{
              "variable": "",
              "components": ["", ""]
            }}
          ],
          "moderators": ["", ""],
          "mediators": ["", ""],
          "other_variables_controls": [
            {{
              "name": "",
              "role": ""
            }}
          ]
        }}
        </output_format>

        <input_variables>
        VARIABLE LIST:
        {step1_json}
        </input_variables>

        <context>
        ABSTRACT:
        {abstract}
        </context>
    """)

    prompt_blocks = [
        {
            "role": "developer",
            "content": developer_content
        },
        {
            "role": "user",
            "content": user_content
        }
    ]

    response_dict = call_openai_with_json_complex(prompt_blocks=prompt_blocks)
    tokens = extract_token_usage(response_dict)
    result = parse_json_response(response_dict)

    return result, tokens


print("Step 2: step2_hierarchy_and_normalize() defined")

In [ ]:
# Cell 7 — Step 3: Relevant Sentence Extraction

def step3_extract_relevant_sentences(
    abstract: str,
    variable_list: List[str]
) -> Tuple[dict, Dict[str, int]]:
    """
    Extract relevant empirical sentences from abstract.
    Uses gpt-4.1 via Responses API (simple model).
    """

    vars_json = json.dumps(variable_list, ensure_ascii=False)

    developer_content = dedent("""\
        You are a scientific-text analyst.

        <rules>
        Your job is to extract ONLY the sentences that report empirical relationships tested or found in this paper.

        Keep only the sentences that explicitly state an empirical relationship and:
        (a) propose it,
        (b) test or estimate it, or
        (c) report findings about it.

        The most critical rule is to keep sentences that describe:
        - what/how this study does
        - what this study tests
        - what this study found

        Exclude sentences describing:
        - previous research has found
        - the literature shows
        - prior work
        - other background information
        - ground truth statements not tied to the current study's method or findings

        INCLUDE these sentence types:
        - Hypotheses / research questions:
          phrases such as "this study", "we hypothesize", "we predict", "we test", "we examine", "we propose", "we argue", "this research examine"
        - Methods / Analysis:
          phrases such as "we argue", "we conducted", "we examine", "we test", "we analyze", "we estimate", "model", "our model introduce", or other sentences that describe the method used to test the relationship between variables
        - Results / findings:
          phrases such as "we find", "results show", "our findings", "we demonstrate", "we argue"

        EXCLUDE these sentence types:
        - Prior work / background:
          sentences mentioning prior research, the literature, prior work, or describing ground truth / phenomena without linking to the current study's method or findings
        - Implications / practical advice:
          sentences such as "we discuss the implications", "our results contribute to the theory", "our findings have implications for"

        CONFLICT RULE:
        If a sentence contains both included and excluded elements, prioritize inclusion.
        Include the sentence if it contains any included element, even if it also contains excluded elements.

        Output only exact sentences from the abstract. Do not paraphrase.
        Return only valid JSON.
        </rules>
    """)

    user_content = dedent(f"""\
        <task>
        Extract only the relevant empirical sentences from this abstract.
        </task>

        <input_variables>
        VARIABLE LIST:
        {vars_json}
        </input_variables>

        <context>
        ABSTRACT:
        {abstract}
        </context>

        <output_format>
        Return a JSON object with this exact structure:

        {{"relevant_sentences": ["The exact first sentence from the abstract.", "The exact second sentence from the abstract."]}}
        </output_format>

        <step_specific_rules>
        - "relevant_sentences" must contain EXACT text from the abstract
        - Do not paraphrase
        - If no relevant sentences are found, return an empty list for "relevant_sentences"

        Return ONLY the JSON object, no additional text.
        </step_specific_rules>
    """)

    prompt_blocks = [
        {
            "role": "developer",
            "content": developer_content
        },
        {
            "role": "user",
            "content": user_content
        }
    ]

    response_dict = call_openai_with_json_simple(prompt_blocks=prompt_blocks)
    tokens = extract_token_usage(response_dict)
    result = parse_json_response(response_dict)

    return result, tokens


print("Step 3: step3_extract_relevant_sentences() defined")

In [ ]:
# Cell 8 — Step 4: Relationship Extraction

def step4_extract_relationships(
    abstract: str,
    variable_list: List[str],
    relevant_sentences: List[str]
) -> Tuple[dict, Dict[str, int]]:
    """
    Extract directional, correlational, and moderation relationships.
    Uses gpt-5.2 via Responses API.
    Receives relevant sentences as additional context.
    """

    variable_list_str = json.dumps(variable_list, ensure_ascii=False)
    sentences_str = json.dumps(relevant_sentences, ensure_ascii=False) if relevant_sentences else "[]"

    developer_content = dedent("""\
        You are an expert scientific-text analyst specializing in extracting empirical relationships from research abstracts.

        <rules>
        VARIABLE LIST ENFORCEMENT
        - Only extract relationships where ALL variables are in the VARIABLE LIST.
        - If a variable is semantically equivalent to one in the list, normalize to the list version.
        - If no close match exists, discard the relationship.

        ONE CATEGORY PER PAIR
        - A variable pair can appear in only one category.
        - If both directional and correlational apply, keep directional.

        USE RELEVANT SENTENCES
        - Ground all relationships primarily in the provided relevant sentences.

        RELATIONSHIP TYPES

        DIRECTIONAL (Var1 -> Var2):
        One variable affects or influences another with clear direction.

        CORRELATIONAL (Var1 <-> Var2):
        Non-directional association with no causal claim.

        MODERATION:
        A third variable modifies the relationship between two others.

        IMPORTANT:
        - Moderation must also include the underlying X-Y relationship.
        - Interaction effects produce both directional and moderation entries.

        VALIDATION STATE
        - validated: supported by results
        - null: tested but not supported
        - hypothesized: proposed but not confirmed

        GENERAL
        - Use exact variable names from VARIABLE LIST
        - Return only valid JSON
        </rules>
    """)

    user_content = dedent(f"""\
        <task>
        Extract all directional, correlational, and moderation relationships from the abstract.
        </task>

        <procedure>
        Step 0: Keep only variables from the VARIABLE LIST.

        Step 1: Use RELEVANT SENTENCES as primary evidence.

        Step 2: Detect moderation first.

        Step 3: Classify relationships:
        - causal → directional
        - non-causal association → correlational
        - conditional relationship → moderation

        Step 4: Assign validation state (validated, null, hypothesized).

        Step 5: Ensure no pair appears in multiple categories.

        Step 6: Remove any relationship using variables outside the list.

        Step 7: Re-check completeness, especially for null effects.
        </procedure>

        <output_format>
        Return a JSON object with this exact structure:

        {{
          "directional": [
            ["source_var", "direction", "target_var", "validated|null|hypothesized"]
          ],
          "correlational": [
            ["var1", "correlation", "var2", "validated|null|hypothesized"]
          ],
          "moderation": [
            {{
              "moderator": "moderator_var",
              "moderated_variables": ["var1", "var2"],
              "validation": "validated|null|hypothesized"
            }}
          ]
        }}
        </output_format>

        <input_variables>
        VARIABLE LIST:
        {variable_list_str}
        </input_variables>

        <context>
        ABSTRACT:
        {abstract}
        </context>

        <evidence>
        RELEVANT SENTENCES:
        {sentences_str}
        </evidence>
    """)

    prompt_blocks = [
        {
            "role": "developer",
            "content": developer_content
        },
        {
            "role": "user",
            "content": user_content
        }
    ]

    response_dict = call_openai_with_json_complex(prompt_blocks=prompt_blocks)
    tokens = extract_token_usage(response_dict)
    result = parse_json_response(response_dict)

    return result, tokens


print("Step 4: step4_extract_relationships() defined")

In [ ]:
# Cell 9 — Step 5.1: Validate

def step5_1_validate_relationships(
    abstract: str,
    step2_hierarchy: str,
    step4_directional: str,
    step4_correlational: str,
    step4_moderation: str,
    variable_list: str
) -> Tuple[dict, Dict[str, int]]:
    """
    Step 5.1: Validate directional, correlational, and moderation relationships.
    Hierarchy is passed through unchanged for Step 5.2.
    Uses gpt-5.2 via Responses API.
    """

    developer_content = dedent("""\
        Your task is to validate variable name and variable relationships (direction/correlation/moderation/hierarchy) using following rules. 
        <rule>ONLY allow for removing edges marked as "hypothesised" in this step. Do NOT remove directional or correlational relationships coded as "validated" or "null" in this step.  </rule>
        <rule>ONLY allow for changing categorisation (direction/correlation) in this step.</rule>
        <definition>
        RELATIONSHIP TYPES
        DIRECTIONAL (Var1 -> Var2)
       Code as directional when the sentence or immediate context explicitly states that Var1 predicts, affects, influences, increases, decreases, causes, explains, contributes to, leads to, results in, enables, determines, drives, mediates or otherwise changes Var2. For example, code A->B (directional) when:
        - A impact/increase/influenced B
        - a neural region A enable a function B
        - past feature A predicts future outcome B
        - Manipulated Independent variable A results in difference in outcome variable B in experiments
        - B mediate the relationship of A and C (A->B->C)

CORRELATIONAL (Var1 <-> Var2)
Code as correlational when the text only states that Var1 and Var2 are associated, correlated, related, linked, connected, coupled, covarying, or different across naturally occurring groups, without explicitly assigning directional or predictor/outcome roles. 
       
        MODERATION:
        A third variable modifies the relationship between other variables. 
        - M impact the strength of the relationship between X and Y: code as M moderates X-Y
        - M impact the mediated relationship between X and Y through M: code as M moderates X-M-Y
        IMPORTANT:
        - Moderation must also include the underlying relationship.
        - Interaction effects produce both directional and moderation entries.
        VALIDATION STATE
        - validated: supported by results
        - null: tested but not supported
        - hypothesized: proposed but not confirmed
        </definition>
    """)

    user_content = dedent(f"""\
        <task> Validate variable name and variable relationships (direction/correlation/moderation/hierarchy) using following rules. </task>
<procedure>
Step 0: Remove any relationship that duplicates a hierarchy edge.
Step 1: Check Validation Labels
-Remove weak relationships labeled "hypothesized" unless the abstract explicitly states them as the primary hypothesis or they are tested in the model.

Step 2: Distinguish Direction vs. Correlation
-Check whether the previous coding incorrectly treated a directional relationship as a correlation, or a correlation as a directional relationship. When a directional/correlational relationship is miscoded, convert it to the correct type instead of deleting it.

If the abstract includes equal evidence for correlation and direction, code as direction.
Step 3: Validate moderation:

Ensure the moderated relationship is correctly specified; it may involve more than two variables. For example, If the abstract says "M moderates the mediated relationship," code M as moderating all variables in that mediated path.
When an abstract EXPLICITLY claim the interaction/interactive effectof A and B significantly impact C, make sure you have extract two directionality A->C, B->C, and two moderation "A moderate B and C", "B moderate A and C". -Check whether Boundary Condition has been mistakenly counted as moderation. For example: The results are validated across different conditions -> not moderation. A moderation requires a comparison across different levels of the moderator. The relationship between A and B are not differed under different level of M -> Moderation (null) Step 4: Check repetitive variable name
Ensure each variable names refers to a distinct variable in the abstract. Identify synonyms and coreference relations. Merge them.
If multiple names refer to the same underlying variable or relationship, merge them as A(B), and update the relationship accordingly. </procedure>
<output_format> Return a JSON object with this exact structure:
{{
"hierarchy": [["upper_var", "hierarchy", "lower_var"]],
"directional": [["source_var", "direction", "target_var", "validated|null|hypothesized"]],
"correlational": [["var1", "correlation", "var2", "validated|null|hypothesized"]],
"moderation": [
{{
"moderator": "mod_var",
"moderated_variables": ["var1", "var2"],
"validation": "validated|null|hypothesized"
}}
],
"justifications": [
{{
"change": "description",
"reason": "based on abstract"
}}
]
}}
</output_format>

<context>
ABSTRACT:
{abstract}
</context>
<input_variables>
VARIABLE LIST:
{variable_list}
</input_variables>
<current_state>
CURRENT HIERARCHY:
{step2_hierarchy}
CURRENT DIRECTIONAL:
{step4_directional}
CURRENT CORRELATIONAL:
{step4_correlational}
CURRENT MODERATION:
{step4_moderation}
</current_state>
    """)

    prompt_blocks = [
        {
            "role": "developer",
            "content": developer_content
        },
        {
            "role": "user",
            "content": user_content
        }
    ]

    response_dict = call_openai_with_json_complex(prompt_blocks=prompt_blocks)
    tokens = extract_token_usage(response_dict)
    result = parse_json_response(response_dict)

    return result, tokens


print("Step 5.1: step5_1_validate_relationships() defined")